# 06 - Metrics, donor uncertainty, and run reporting

- **Meeting item:** Metrics quality
- **Commit:** C08 - `feat: add benchmark metrics and donor bootstrap`
- **Commit achievement:** adds donor-level uncertainty and valid comparison statistics.
- **Data mode:** `synthetic`
- **Evidence labels used:** verified, unknown
- **Human gate:** is every estimate reported with an interval, a valid replicate count, and an explicit reason when a metric does not apply?

This notebook measures nothing about biology. It checks that the measurement layer
behaves: intervals come from resampling donors rather than cells, and a metric that
cannot be computed says so instead of returning a number-shaped NaN.

Predictions below come from a nearest-class-centroid rule fitted on training cells only.
No model is trained here; the rule exists to produce a realistic prediction table.

In [ ]:
import numpy as np
import pandas as pd

import p22
from p22.data import make_donor_split
from p22.eval import (
    accuracy,
    balanced_accuracy,
    cohens_h,
    compute_metrics,
    donor_bootstrap,
    macro_f1,
    macro_ovr_auroc,
    mcnemar_test,
    multiclass_ece,
    summarize_seeds,
)
from p22.testing import make_synthetic_multimodal

pd.set_option("display.width", 120)

print("approval state blocked:", p22.approval_blocked())
print("data mode:", p22.SYNTHETIC)

dataset = make_synthetic_multimodal(n_donors=18, cells_per_donor=24, seed=12)
split = make_donor_split(dataset.donor_ids, labels=dataset.labels, seed=12)
print("donors per split:", split.summary["realized_donor_counts"])
print("cells train/test:", split.train_index.size, split.test_index.size)

## 1. A prediction table from train-fitted centroids (evidence label: verified)

Centroids are computed on training cells only. Class scores are negative squared
distances; probabilities are the softmax of those scores. Both views are concatenated
for the two-view rule, and view A alone for the single-view rule.

In [ ]:
def centroid_predictions(features, labels, train_index, eval_index, n_classes, temperature=1.0):
    """Predict with class centroids fitted on train cells only."""
    centroids = np.stack(
        [features[train_index][labels[train_index] == label].mean(axis=0) for label in range(n_classes)]
    )
    distances = ((features[eval_index][:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    scores = -distances / (temperature * features.shape[1])
    scores = scores - scores.max(axis=1, keepdims=True)
    weights = np.exp(scores)
    probabilities = weights / weights.sum(axis=1, keepdims=True)
    return probabilities.argmax(axis=1), probabilities


n_classes = int(dataset.labels.max()) + 1
both_views = np.hstack([dataset.view_a, dataset.view_b])
test_index = split.test_index
y_test = dataset.labels[test_index]
donors_test = dataset.donor_ids[test_index]

pred_two, prob_two = centroid_predictions(
    both_views, dataset.labels, split.train_index, test_index, n_classes
)
pred_one, prob_one = centroid_predictions(
    dataset.view_a, dataset.labels, split.train_index, test_index, n_classes
)

print("test cells:", y_test.size, "| test donors:", len(set(donors_test)))
print("two-view accuracy:", round(float((pred_two == y_test).mean()), 4))
print("view-A-only accuracy:", round(float((pred_one == y_test).mean()), 4))

## 2. Metric set with explicit applicability (evidence label: verified)

Each metric returns a value or a reason. Nothing returns NaN.

In [ ]:
rows = []
for name, pred, prob in (("two_view", pred_two, prob_two), ("view_a_only", pred_one, prob_one)):
    for metric_name, result in compute_metrics(y_test, pred, prob, n_classes=n_classes).items():
        rows.append(
            {
                "model": name,
                "metric": metric_name,
                "value": None if result.value is None else round(result.value, 4),
                "applicable": result.applicable,
                "not_applicable": result.not_applicable,
            }
        )

metric_table = pd.DataFrame(rows)
assert metric_table["value"].apply(lambda v: v is None or np.isfinite(v)).all()
metric_table

## 3. Metrics that refuse to answer (evidence label: verified)

Three ways an estimate can be undefined: a class missing from the evaluation labels,
a single-class evaluation set, and probabilities that were never produced. Each returns
a reason a reviewer can read.

In [ ]:
refusals = []

single_class = np.zeros(20, dtype=int)
refusals.append(("balanced_accuracy, one class in y_true", balanced_accuracy(single_class, single_class)))

two_of_three = y_test[y_test < 2][:40]
prob_subset = prob_two[y_test < 2][:40]
refusals.append(
    ("macro_ovr_auroc, class 2 absent from y_true", macro_ovr_auroc(two_of_three, prob_subset, n_classes=n_classes))
)

no_probabilities = compute_metrics(y_test, pred_two)
refusals.append(("macro_ovr_auroc, no probabilities supplied", no_probabilities["macro_ovr_auroc"]))
refusals.append(("multiclass_ece, no probabilities supplied", no_probabilities["multiclass_ece"]))

refusal_table = pd.DataFrame(
    [
        {"situation": situation, "value": result.value, "not_applicable": result.not_applicable}
        for situation, result in refusals
    ]
)
assert refusal_table["value"].isna().all()
assert refusal_table["not_applicable"].str.len().gt(0).all()
refusal_table

## 4. Calibration is reported separately from accuracy (evidence label: verified)

Expected calibration error compares top-label confidence with top-label accuracy. A rule
can be accurate and overconfident at the same time, so the two numbers are kept apart.

In [ ]:
def one_hot(labels, n):
    matrix = np.zeros((labels.size, n))
    matrix[np.arange(labels.size), labels] = 1.0
    return matrix


calibration_rows = [
    {
        "predictions": "centroid softmax (two views)",
        "accuracy": round(accuracy(y_test, pred_two).value, 4),
        "ece": round(multiclass_ece(y_test, prob_two).value, 4),
    },
    {
        "predictions": "same predictions, forced to full confidence",
        "accuracy": round(accuracy(y_test, pred_two).value, 4),
        "ece": round(multiclass_ece(y_test, one_hot(pred_two, n_classes)).value, 4),
    },
    {
        "predictions": "perfect and honest",
        "accuracy": 1.0,
        "ece": round(multiclass_ece(y_test, one_hot(y_test, n_classes)).value, 4),
    },
]
calibration = pd.DataFrame(calibration_rows)
print("forced confidence leaves accuracy unchanged and raises calibration error:")
calibration

## 5. Donor bootstrap, and why cell resampling understates spread (evidence label: verified)

The donor bootstrap draws whole donors with replacement. The comparison row below pretends
each cell is its own donor, which is the mistake this module exists to prevent: it produces a
visibly narrower interval on the same predictions.

In [ ]:
boot_rows = []
for metric_fn, label in ((accuracy, "accuracy"), (balanced_accuracy, "balanced_accuracy"), (macro_f1, "macro_f1")):
    result = donor_bootstrap(metric_fn, y_test, pred_two, donors_test, n_replicates=400, seed=3)
    boot_rows.append(
        {
            "metric": label,
            "unit": result.unit,
            "estimate": round(result.estimate, 4),
            "lower": round(result.lower, 4),
            "upper": round(result.upper, 4),
            "width": round(result.upper - result.lower, 4),
            "valid": result.n_valid,
            "failed": result.n_failed,
        }
    )

fake_cell_donors = np.array([f"cell_{i}" for i in range(y_test.size)])
cell_level = donor_bootstrap(accuracy, y_test, pred_two, fake_cell_donors, n_replicates=400, seed=3)
boot_rows.append(
    {
        "metric": "accuracy",
        "unit": "cell (wrong unit, shown for contrast)",
        "estimate": round(cell_level.estimate, 4),
        "lower": round(cell_level.lower, 4),
        "upper": round(cell_level.upper, 4),
        "width": round(cell_level.upper - cell_level.lower, 4),
        "valid": cell_level.n_valid,
        "failed": cell_level.n_failed,
    }
)

bootstrap_table = pd.DataFrame(boot_rows)
donor_width = bootstrap_table.iloc[0]["width"]
cell_width = bootstrap_table.iloc[-1]["width"]
print(f"donor-unit width {donor_width} vs cell-unit width {cell_width}")
assert donor_width > cell_width
bootstrap_table

## 6. A partly failed bootstrap says how many replicates failed (evidence label: verified)

Here one donor holds every minority-class cell. Replicates that miss that donor cannot
compute balanced accuracy. The result keeps the estimate, counts the failures, and lists
their reasons rather than averaging over what survived and calling it clean.

In [ ]:
rare_donors = np.concatenate([np.repeat([f"donor_{i:02d}" for i in range(4)], 10), np.repeat("donor_rare", 10)])
rare_true = np.concatenate([np.zeros(40, dtype=int), np.ones(10, dtype=int)])
rare_pred = rare_true.copy()

fragile = donor_bootstrap(balanced_accuracy, rare_true, rare_pred, rare_donors, n_replicates=200, seed=9)
print("estimate:", fragile.estimate)
print("interval:", (round(fragile.lower, 4), round(fragile.upper, 4)))
print("requested / valid / failed:", fragile.n_replicates_requested, fragile.n_valid, fragile.n_failed)
print("failure reasons:", list(fragile.failure_reasons))
assert fragile.n_valid + fragile.n_failed == fragile.n_replicates_requested
assert fragile.n_failed > 0

single_donor = donor_bootstrap(accuracy, rare_true[:10], rare_pred[:10], np.repeat("donor_only", 10), n_replicates=50)
print()
print("one donor available ->", single_donor.not_applicable)
assert single_donor.lower is None

## 7. Paired comparison and effect size (evidence label: verified)

McNemar compares two rules on the same cells and only uses the cells where they disagree.
Cohen's h reports how far apart the two accuracies are, so a small p-value on a tiny
difference cannot pass as an important one.

In [ ]:
correct_two = pred_two == y_test
correct_one = pred_one == y_test

paired = mcnemar_test(correct_two, correct_one)
effect = cohens_h(float(correct_two.mean()), float(correct_one.mean()))
print("only two-view correct:", paired.n_only_a_correct)
print("only view-A correct:", paired.n_only_b_correct)
print("statistic:", None if paired.statistic is None else round(paired.statistic, 4))
print("p value:", None if paired.p_value is None else round(paired.p_value, 6))
print("not applicable:", paired.not_applicable)
print("cohens h:", round(effect, 4))

identical = mcnemar_test(correct_two, correct_two)
print()
print("comparing a rule with itself ->", identical.not_applicable)
assert identical.p_value is None

## 8. Five-seed summary (evidence label: verified)

Repeating the whole pipeline over five seeds gives an across-seed spread. One seed is not
a result; the summary refuses to report a spread from a single value.

In [ ]:
seed_rows = []
for seed in (11, 12, 13, 14, 15):
    seed_data = make_synthetic_multimodal(n_donors=15, cells_per_donor=24, seed=seed)
    seed_split = make_donor_split(seed_data.donor_ids, labels=seed_data.labels, seed=seed)
    seed_features = np.hstack([seed_data.view_a, seed_data.view_b])
    seed_pred, seed_prob = centroid_predictions(
        seed_features, seed_data.labels, seed_split.train_index, seed_split.test_index, n_classes
    )
    seed_true = seed_data.labels[seed_split.test_index]
    metrics = compute_metrics(seed_true, seed_pred, seed_prob, n_classes=n_classes)
    seed_rows.append({"seed": seed, **{name: result.value for name, result in metrics.items()}})

seed_table = pd.DataFrame(seed_rows).round(4)
summaries = pd.DataFrame(
    [
        summarize_seeds(seed_table[column].tolist(), metric=column).to_dict()
        for column in seed_table.columns
        if column != "seed"
    ]
)
print(seed_table.to_string(index=False))
print()
print("single seed only ->", summarize_seeds([seed_table["accuracy"].iloc[0]]).not_applicable)
summaries.drop(columns=["level"])

## 9. Human gate

Answer before moving on: is every estimate reported with an interval, a valid replicate
count, and an explicit reason when a metric does not apply?

In [ ]:
checks = {
    "every metric value is a number or an explicit refusal, never NaN": bool(
        metric_table["value"].apply(lambda v: v is None or np.isfinite(v)).all()
        and refusal_table["not_applicable"].str.len().gt(0).all()
    ),
    "bootstrap resamples donors, not cells": all(row == "donor" for row in bootstrap_table["unit"][:3]),
    "cell-unit resampling gives a narrower, over-confident interval": bool(donor_width > cell_width),
    "replicate accounting adds up (valid + failed = requested)": bool(
        fragile.n_valid + fragile.n_failed == fragile.n_replicates_requested
    ),
    "failed replicates carry readable reasons": len(fragile.failure_reasons) > 0,
    "too few donors blocks the interval instead of faking one": single_donor.lower is None,
    "paired test refuses when two rules never disagree": identical.p_value is None,
    "effect size accompanies the paired p value": np.isfinite(effect),
    "five-seed spread reported; one seed refused": bool(
        summaries["n_seeds"].eq(5).all()
        and summarize_seeds([0.8]).not_applicable is not None
    ),
    "no disease data, no network calls, approval still blocked": p22.approval_blocked(),
}
for description, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {description}")
assert all(checks.values()), "C08 gate not satisfied"
print()
print("C08 gate: PASS")